# 03 — Publish results to the databases and query them

The pipeline (`02_carbon_pipeline.ipynb`) writes plain files to `IIRS/Carbon_Stocks/results/`, and
the dashboard runs on those alone (`DB_MODE=files`). This notebook loads the
same results into two databases, checks that nothing was lost on the way, and
answers questions that are natural in a database and awkward in a CSV:

| Store | Holds | Used for |
|---|---|---|
| PostgreSQL + PostGIS `grid_cells` | one polygon per cell with its carbon values, area and sources, GiST-indexed | map tiles by bounding box (`DB_MODE=local`), spatial joins |
| PostGIS `district_boundaries` | the two district polygons | totals inside a boundary, computed in SQL |
| PostgreSQL `district_summary` | one row per pipeline run, headline + full JSON | run history |
| MongoDB `district_summary`, `model_metrics`, `ndvi_monthly`, `cells` | the result documents and the per-cell table | dashboard API, aggregation pipelines |

It copies; it never computes a published number. Every table is rebuilt from
scratch, so re-running is safe. Each store is tried on its own: if MongoDB is
not running, PostgreSQL is still loaded, and the reverse.

**Setup (Windows).** Install PostgreSQL 16 with the PostGIS bundle (Stack
Builder → Spatial Extensions) and MongoDB Community Server; start both; copy
`.env.example` to `.env` and set `PG_PASSWORD`. Then run this notebook and set
`DB_MODE=local` in `.env` to make the dashboard read from the databases.


In [1]:
import json
import os
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd
from dotenv import load_dotenv

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(ROOT))
import config as C

load_dotenv(ROOT / ".env")
summary = json.loads((C.RESULTS / "district_summary.json").read_text())
metrics = json.loads((C.RESULTS / "model_metrics.json").read_text())
ndvi = json.loads((C.RESULTS / "ndvi_monthly.json").read_text())
csv = C.RESULTS / "carbon_cells.csv"          # git-ignored; the committed parquet holds the same cells
cells = pd.read_csv(csv) if csv.exists() else pd.read_parquet(C.RESULTS / "carbon_cells.parquet")
if "WKT" not in cells:
    cells["WKT"] = C.lattice_geometry(cells["grid_row"], cells["grid_col"])["WKT"].to_numpy()
C.assert_unique(cells)
bounds = {name: C.load_boundary(name) for name in [C.BOUNDARY_DEFAULT, C.BOUNDARY_CHECK]}
HEADLINE_SOIL = summary["soil"]["stock_mtc"]["estimate"]
HEADLINE_NPP = summary["npp"]["flux_mtc_per_year"]["estimate"]
print(f"run {summary['run_date']} ({summary['mode']}): {len(cells):,} cells | "
      f"soil {HEADLINE_SOIL} MtC | NPP {HEADLINE_NPP} MtC/yr")


run 2026-10-08 (v2-census): 71,197 cells | soil 10.5371 MtC | NPP 0.3353 MtC/yr


## Loaders

In [2]:
# Columns the dashboard reads (names as in carbon_cells.csv -> names in PostgreSQL)
CELL_FIELDS = ["cell_id", "lon", "lat", "cell_area_ha", "agri_class", "Ag/NonAg_m", "DEM_mean", "Slope_mean",
               "soil_status", "w_soil", "bd_gcm3_rec", "soc_gkg", "soc_stock_tc_ha", "soc_stock_tc", "soc_source",
               "npp_flux_tc_ha_yr", "npp_flux_tc_yr", "npp_source"]
PG_COLS = ["cell_id", "lon", "lat", "cell_area_ha", "agri_class", "agnonag_m", "dem_mean", "slope_mean",
           "soil_status", "w_soil", "bd_gcm3_rec", "soc_gkg", "soc_stock_tc_ha", "soc_stock_tc", "soc_source",
           "npp_flux_tc_ha_yr", "npp_flux_tc_yr", "npp_source"]
CELL_FIELDS = [c for c in CELL_FIELDS if c in cells.columns]
PG_COLS = [p for c, p in zip(["cell_id", "lon", "lat", "cell_area_ha", "agri_class", "Ag/NonAg_m", "DEM_mean",
                              "Slope_mean", "soil_status", "w_soil", "bd_gcm3_rec", "soc_gkg", "soc_stock_tc_ha",
                              "soc_stock_tc", "soc_source", "npp_flux_tc_ha_yr", "npp_flux_tc_yr", "npp_source"],
                             PG_COLS) if c in CELL_FIELDS]
TEXT_COLS = {"cell_id", "soil_status", "soc_source", "npp_source"}


def clean(v):
    return None if pd.isna(v) else (v.item() if hasattr(v, "item") else v)


def publish_mongo(db):
    for name, doc in [("district_summary", summary), ("model_metrics", metrics)]:
        db[name].drop()
        db[name].insert_one(dict(doc))
    db["ndvi_monthly"].drop()
    db["ndvi_monthly"].insert_many([dict(d) for d in ndvi])
    db["cells"].drop()
    recs = [{k: clean(v) for k, v in r.items()} for r in cells[CELL_FIELDS].to_dict(orient="records")]
    for i in range(0, len(recs), 10_000):
        db["cells"].insert_many(recs[i:i + 10_000])
    db["cells"].create_index("cell_id", unique=True)
    return {n: db[n].count_documents({}) for n in ["district_summary", "model_metrics", "ndvi_monthly", "cells"]}


def publish_postgres(conn):
    from psycopg2.extras import Json, execute_values

    cur = conn.cursor()
    cur.execute("CREATE EXTENSION IF NOT EXISTS postgis")
    cur.execute("DROP TABLE IF EXISTS grid_cells")
    coldefs = ",\n".join(f"{c} {'TEXT' if c in TEXT_COLS else 'DOUBLE PRECISION'}" for c in PG_COLS if c != "cell_id")
    cur.execute(f"""CREATE TABLE grid_cells (
                        cell_id TEXT PRIMARY KEY,
                        geom    GEOMETRY(Polygon, 4326) NOT NULL,
                        {coldefs})""")
    rows = [tuple(clean(r[c]) for c in CELL_FIELDS) + (r["WKT"],) for r in cells.to_dict(orient="records")]
    execute_values(cur, f"INSERT INTO grid_cells ({', '.join(PG_COLS)}, geom) VALUES %s", rows,
                   template="(" + ", ".join(["%s"] * len(PG_COLS)) + ", ST_GeometryN(ST_GeomFromText(%s, 4326), 1))",
                   page_size=2000)
    cur.execute("CREATE INDEX grid_cells_geom_idx ON grid_cells USING GIST (geom)")

    cur.execute("DROP TABLE IF EXISTS district_boundaries")
    cur.execute("CREATE TABLE district_boundaries (name TEXT PRIMARY KEY, geom GEOMETRY(MultiPolygon, 4326) NOT NULL)")
    for name, shape in bounds.items():
        cur.execute("INSERT INTO district_boundaries VALUES (%s, ST_Multi(ST_GeomFromText(%s, 4326)))", (name, shape.wkt))
    cur.execute("CREATE INDEX district_boundaries_geom_idx ON district_boundaries USING GIST (geom)")

    cur.execute("""
        CREATE TABLE IF NOT EXISTS district_summary (
            id SERIAL PRIMARY KEY, run_date DATE NOT NULL, mode TEXT NOT NULL,
            soil_stock_mtc DOUBLE PRECISION, soil_stock_ci_lo DOUBLE PRECISION, soil_stock_ci_hi DOUBLE PRECISION,
            npp_flux_mtc_yr DOUBLE PRECISION, npp_flux_ci_lo DOUBLE PRECISION, npp_flux_ci_hi DOUBLE PRECISION,
            payload JSONB NOT NULL)""")
    s, f = summary["soil"]["stock_mtc"], summary["npp"]["flux_mtc_per_year"]
    ci = lambda e, i: e["ci95"][i] if e.get("ci95") else None
    cur.execute("DELETE FROM district_summary WHERE run_date = %s AND mode = %s", (summary["run_date"], summary["mode"]))
    cur.execute("""INSERT INTO district_summary (run_date, mode, soil_stock_mtc, soil_stock_ci_lo, soil_stock_ci_hi,
                   npp_flux_mtc_yr, npp_flux_ci_lo, npp_flux_ci_hi, payload) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)""",
                (summary["run_date"], summary["mode"], s["estimate"], ci(s, 0), ci(s, 1),
                 f["estimate"], ci(f, 0), ci(f, 1), Json(summary)))
    conn.commit()
    cur.execute("SELECT count(*), count(DISTINCT cell_id) FROM grid_cells")
    n, n_unique = cur.fetchone()
    assert n == n_unique == len(cells)
    return {"grid_cells": n, "district_boundaries": len(bounds)}


## Connect and load (`DB_MODE=local`)

In [3]:
from pymongo import MongoClient
import psycopg2

loaded, skipped = {}, {}

def pg_connect(prefix="PG", ssl="prefer"):
    if not os.getenv(f"{prefix}_PASSWORD"):
        raise ValueError(f"{prefix}_PASSWORD missing from .env")
    return psycopg2.connect(host=os.getenv(f"{prefix}_HOST", "localhost"), port=int(os.getenv(f"{prefix}_PORT", "5432")),
                            dbname=os.getenv(f"{prefix}_DB", "postgres"), user=os.getenv(f"{prefix}_USER", "postgres"),
                            password=os.getenv(f"{prefix}_PASSWORD"), sslmode=ssl, connect_timeout=5)

try:
    conn = pg_connect()
    loaded["postgres"] = publish_postgres(conn)
    print("PostgreSQL:", loaded["postgres"])
except Exception as e:
    conn = None
    skipped["PostgreSQL"] = f"{e.__class__.__name__}: {str(e).strip()[:300]}"
    print(f"PostgreSQL skipped: {skipped['PostgreSQL']}")

try:
    mongo = MongoClient(os.getenv("MONGO_URI", "mongodb://localhost:27017/"), serverSelectionTimeoutMS=5000)
    mongo.admin.command("ping")
    mdb = mongo[os.getenv("MONGO_DB", "carbon_stock_ludhiana")]
    loaded["mongo"] = publish_mongo(mdb)
    print("MongoDB:", loaded["mongo"])
except Exception as e:
    mdb = None
    skipped["MongoDB"] = f"{e.__class__.__name__}: {str(e).strip()[:300]}"
    print(f"MongoDB skipped: {skipped['MongoDB']}")

if not loaded:
    print("\nNo database reachable. The dashboard still works with DB_MODE=files.")


PostgreSQL: {'grid_cells': 71197, 'district_boundaries': 2}


MongoDB skipped: ServerSelectionTimeoutError: localhost:27017: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 5.0s, Topology Description: <TopologyDescription id: 6ac7c7382a033b5d4517d262, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27017) server_typ


## Verify, then query

The database totals must reproduce the published headline exactly: that is
the check that the load lost nothing. The queries below then show what each
store is good for.


In [4]:
if conn is not None:
    cur = conn.cursor()
    cur.execute("SELECT sum(soc_stock_tc) / 1e6, sum(npp_flux_tc_yr) / 1e6, sum(cell_area_ha) FROM grid_cells")
    soil_db, npp_db, area_db = cur.fetchone()
    print(f"PostGIS sums: soil {soil_db:.4f} MtC (published {HEADLINE_SOIL}) | NPP {npp_db:.4f} MtC/yr "
          f"(published {HEADLINE_NPP}) | area {area_db:,.0f} ha")
    if summary["mode"] == "v2-census":
        assert abs(soil_db - HEADLINE_SOIL) < 1e-3 and abs(npp_db - HEADLINE_NPP) < 1e-3, "database totals differ from results"

    # Spatial join, recomputed from geometry alone: stock inside each boundary =
    # density x valid-soil share x geodesic area of (cell ∩ boundary). Independent of the
    # boundary weights the pipeline used, so it must reproduce both published variants.
    cur.execute('''
        SELECT b.name, count(*) AS cells,
               round((sum(g.soc_stock_tc_ha * g.w_soil *
                          ST_Area(ST_Intersection(g.geom, b.geom)::geography)) / 1e4 / 1e6)::numeric, 3) AS soil_mtc,
               round((ST_Area(b.geom::geography) / 1e4)::numeric) AS boundary_ha
        FROM district_boundaries b
        JOIN grid_cells g ON ST_Intersects(b.geom, g.geom)
        WHERE g.soc_stock_tc_ha IS NOT NULL
        GROUP BY b.name, b.geom ORDER BY b.name''')
    sj = pd.DataFrame(cur.fetchall(), columns=["boundary", "cells touching", "soil MtC (SQL)", "boundary ha"])
    def boundary_of(label):                      # sensitivity labels name the boundary by source
        low = label.lower()
        return ("ludhiana_geoboundaries_adm2" if "geoboundaries" in low else
                "ludhiana_census2011_datameet" if ("census" in low or "datameet" in low) else None)
    published = {boundary_of(v["variant"]): v["estimate"] for v in summary["soil"].get("sensitivity", [])
                 if v["variant"].startswith("Boundary")}
    sj["published MtC"] = sj["boundary"].map(published)
    print(sj.to_string(index=False))

    # Mean stock density by soil status and data source
    cur.execute('''
        SELECT soil_status, soc_source, count(*) AS cells,
               round((sum(soc_stock_tc) / NULLIF(sum(cell_area_ha * w_soil), 0))::numeric, 2) AS t_per_ha_valid_soil
        FROM grid_cells GROUP BY 1, 2 ORDER BY 3 DESC''')
    print(pd.DataFrame(cur.fetchall(), columns=["soil status", "source", "cells", "tC/ha of valid soil"]).to_string(index=False))

    # The dashboard's map query: cells in a 5 km box around Ludhiana city centre, served from the GiST index
    cur.execute("EXPLAIN SELECT cell_id FROM grid_cells WHERE geom && ST_MakeEnvelope(75.83, 30.88, 75.88, 30.93, 4326)")
    print("\n" + "\n".join(r[0] for r in cur.fetchall()[:3]))
    conn.close()


PostGIS sums: soil 10.5371 MtC (published 10.5371) | NPP 0.3353 MtC/yr (published 0.3353) | area 369,961 ha


                    boundary  cells touching soil MtC (SQL) boundary ha  published MtC
ludhiana_census2011_datameet           65138         10.183      358482        10.1832
 ludhiana_geoboundaries_adm2           67233         10.537      369961        10.5371
soil status       source  cells tC/ha of valid soil
       full     observed  62244               30.84
    partial     observed   6044               29.59
     nodata no soil data   2585                None
    partial no soil data    324                0.00

Bitmap Heap Scan on grid_cells  (cost=4.33..31.38 rows=7 width=32)
  Recheck Cond: (geom && '0103000020E6100000010000000500000085EB51B81EF55240E17A14AE47E13E4085EB51B81EF55240AE47E17A14EE3E40B81E85EB51F85240AE47E17A14EE3E40B81E85EB51F85240E17A14AE47E13E4085EB51B81EF55240E17A14AE47E13E40'::geometry)
  ->  Bitmap Index Scan on grid_cells_geom_idx  (cost=0.00..4.33 rows=7 width=0)


In [5]:
if mdb is not None:
    pipeline = [
        {"$group": {"_id": "$soc_source", "cells": {"$sum": 1}, "soil_tc": {"$sum": "$soc_stock_tc"},
                    "npp_tc_yr": {"$sum": "$npp_flux_tc_yr"}}},
        {"$sort": {"cells": -1}},
    ]
    agg = pd.DataFrame(list(mdb["cells"].aggregate(pipeline))).rename(columns={"_id": "soc_source"})
    agg["soil_mtc"], agg["npp_mtc_yr"] = agg.pop("soil_tc") / 1e6, agg.pop("npp_tc_yr") / 1e6
    print(agg.round(4).to_string(index=False))
    total = agg["soil_mtc"].sum()
    print(f"MongoDB soil total {total:.4f} MtC (published {HEADLINE_SOIL})")
    if summary["mode"] == "v2-census":
        assert abs(total - HEADLINE_SOIL) < 1e-3
    mongo.close()


## Cloud (`DB_MODE=cloud`, optional) — skipped unless the credentials are in `.env`

In [6]:
if os.getenv("ATLAS_URI"):
    atlas = MongoClient(os.getenv("ATLAS_URI"), serverSelectionTimeoutMS=10_000)
    print("Atlas:", publish_mongo(atlas[os.getenv("MONGO_DB", "carbon_stock_ludhiana")]))
    atlas.close()
else:
    print("ATLAS_URI not set; skipped")

if os.getenv("SUPABASE_PASSWORD"):
    sconn = pg_connect("SUPABASE", ssl="require")
    try:
        print("Supabase:", publish_postgres(sconn))
    finally:
        sconn.close()
else:
    print("SUPABASE_PASSWORD not set; skipped")


ATLAS_URI not set; skipped
SUPABASE_PASSWORD not set; skipped


## Result

One line per store. With `DB_MODE=local` in `.env`, a store that was skipped
stops the notebook with a red error, because the dashboard would otherwise fall
back to the files (`data: files-fallback`) without saying why.

In [7]:
print("PostgreSQL :", f"loaded {loaded['postgres']}" if "postgres" in loaded else f"NOT loaded - {skipped.get('PostgreSQL')}")
print("MongoDB    :", f"loaded {loaded['mongo']}" if "mongo" in loaded else f"NOT loaded - {skipped.get('MongoDB')}")
if os.getenv("DB_MODE", "files") in ("local", "cloud") and skipped:
    raise RuntimeError("DB_MODE=%s but not loaded: %s. Fix the error above (password and PG_* values in .env, "
                       "PostGIS installed, server running) and run this notebook again." % (os.getenv("DB_MODE"), ", ".join(skipped)))

PostgreSQL : loaded {'grid_cells': 71197, 'district_boundaries': 2}
MongoDB    : NOT loaded - ServerSelectionTimeoutError: localhost:27017: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 5.0s, Topology Description: <TopologyDescription id: 6ac7c7382a033b5d4517d262, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27017) server_typ
